<a href="https://colab.research.google.com/github/lidwaa/cinetrack/blob/main/ocr_benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 OCR Benchmark Dashboard
Ce notebook charge tous vos fichiers `results_nomduparseur.csv` depuis un dossier,
agrège les données, calcule des stats et génère un dashboard HTML interactif.

## ⚙️ Configuration

In [ ]:
# ─── PARAMÈTRES À MODIFIER ───────────────────────────────────────────────
CSV_FOLDER  = "./csv_results"      # chemin vers votre dossier de CSV
OUTPUT_HTML = "./ocr_dashboard.html"  # fichier HTML généré

# ─── SPECS TECHNIQUES DU BENCHMARK ──────────────────────────────────────
# Remplissez les champs que vous voulez voir apparaître dans le footer.
# Laissez une valeur vide "" pour masquer un champ.
BENCHMARK_SPECS = {
    # ── Environnement ──────────────────────────────────────────────────
    "date":            "2026-04-24",
    "os":              "Ubuntu 22.04 LTS",
    "python_version":  "3.11.4",
    "cpu":             "Intel Xeon E5-2690 v4 × 28 cores",
    "gpu":             "NVIDIA A100 40GB",
    "ram":             "128 GB",

    # ── Pipeline ───────────────────────────────────────────────────────
    "pipeline_version": "v1.2.0",
    "dataset":          "Internal dataset — 320 documents",
    "dataset_split":    "80% train / 20% test",
    "eval_metric":      "GT Accuracy, Fill Rate, LLM Judge Score, Latency",
    "n_runs":           "3 runs averaged",

    # ── Packages Python ────────────────────────────────────────────────
    # Format : "nom_package": "version"
    "packages": {
        "pandas":         "2.2.1",
        "numpy":          "1.26.4",
        "mistralai":      "1.0.3",
        "openai":         "1.23.0",
        "paddleocr":      "2.7.3",
        "pillow":         "10.3.0",
        "pdfplumber":     "0.11.0",
        "transformers":   "4.40.1",
        "torch":          "2.3.0",
    },

    # ── Modèles OCR testés ─────────────────────────────────────────────
    "ocr_models": {
        "markildown":  "Markdown-based layout parser",
        "liteparse":   "Lightweight heuristic parser",
        "paddleocr":   "PaddleOCR v2.7 — PP-OCRv4",
        "litepaddle":  "PaddleOCR lite variant",
        "mistral_ocr": "Mistral OCR API — mistral-ocr-latest",
    },

    # ── Parseurs LLM testés ────────────────────────────────────────────
    "llm_parsers": {
        "mistral_medium": "mistral-medium-2505",
        "gemma_4":        "gemma-3-27b-it",
        "gpt4o":          "gpt-4o-2024-11-20",
    },

    # ── Notes libres ───────────────────────────────────────────────────
    "notes": "Temperature=0 for all LLM parsers. Timeout=60s per document.",
}
# ─────────────────────────────────────────────────────────────────────────

## 📦 Imports

In [ ]:
import os
import re
import json
import warnings
from pathlib import Path

import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')
print('✅ Imports OK')

## 📂 Chargement des CSV

In [ ]:
def load_benchmark_folder(folder: str) -> pd.DataFrame:
    """
    Charge tous les fichiers results_*.csv du dossier.
    Extrait le nom du parseur depuis le nom du fichier.
    Retourne un DataFrame agrégé.
    """
    folder = Path(folder)
    assert folder.exists(), f"Dossier introuvable : {folder.resolve()}"

    pattern = re.compile(r'results[_-](.+)\.csv$', re.IGNORECASE)
    dfs = []

    csv_files = sorted(folder.glob('*.csv'))
    assert len(csv_files) > 0, f"Aucun fichier CSV trouvé dans {folder.resolve()}"

    for filepath in csv_files:
        match = pattern.match(filepath.name)
        parser_name = match.group(1) if match else filepath.stem

        df = pd.read_csv(filepath)
        df.columns = df.columns.str.strip()
        df['parser'] = parser_name
        df['source_file'] = filepath.name
        dfs.append(df)
        print(f'  ✅  {filepath.name:<45} → parseur: "{parser_name}"  ({len(df)} lignes)')

    combined = pd.concat(dfs, ignore_index=True)
    print(f'\n📊 Total : {len(combined)} entrées · {combined["parser"].nunique()} parseur(s) · {combined["model"].nunique()} modèle(s) OCR')
    return combined


print(f'Chargement depuis : {Path(CSV_FOLDER).resolve()}\n')
df = load_benchmark_folder(CSV_FOLDER)
df.head()

## 🔍 Aperçu & validation des données

In [ ]:
EXPECTED_COLS = ['model', 'avg_latency_s', 'fill_rate_pct', 'llm_judge_score_pct', 'gt_accuracy_pct']

print('=== Colonnes détectées ===')
print(list(df.columns))

missing = [c for c in EXPECTED_COLS if c not in df.columns]
if missing:
    print(f'\n⚠️  Colonnes manquantes : {missing}')
    print('   Le dashboard ignorera ces métriques.')
else:
    print('\n✅ Toutes les colonnes attendues sont présentes.')

print('\n=== Valeurs manquantes ===')
print(df[EXPECTED_COLS].isnull().sum())

print('\n=== Types ===')
print(df[EXPECTED_COLS].dtypes)

## 📈 Stats résumé par parseur

In [ ]:
metrics = [c for c in ['gt_accuracy_pct', 'fill_rate_pct', 'llm_judge_score_pct', 'avg_latency_s'] if c in df.columns]

summary = df.groupby('parser')[metrics].agg(['mean', 'min', 'max']).round(2)
summary.columns = ['_'.join(c) for c in summary.columns]

print('=== Résumé par parseur ===')
display(summary)

print('\n=== Meilleur modèle OCR par parseur (GT Accuracy) ===')
if 'gt_accuracy_pct' in df.columns:
    best = df.loc[df.groupby('parser')['gt_accuracy_pct'].idxmax(), ['parser', 'model', 'gt_accuracy_pct', 'avg_latency_s']]
    display(best.reset_index(drop=True))

## 🏆 Classement global

In [ ]:
if 'gt_accuracy_pct' in df.columns:
    ranking = (
        df.groupby(['model', 'parser'])
        .agg({m: 'mean' for m in metrics})
        .round(2)
        .sort_values('gt_accuracy_pct', ascending=False)
        .reset_index()
    )
    ranking.index = ranking.index + 1
    ranking.index.name = 'rank'
    print('=== Classement global (par GT Accuracy décroissant) ===')
    display(ranking)

## 🌐 Génération du Dashboard HTML

In [ ]:
def generate_dashboard(df: pd.DataFrame, output_path: str, specs: dict = None):
    """
    Sérialise les données + specs en JSON et les injecte dans le template HTML.
    Écrit le fichier HTML final.
    """
    records = df.where(pd.notnull(df), None).to_dict(orient='records')
    data_json  = json.dumps(records, ensure_ascii=False)
    specs_json = json.dumps(specs or {}, ensure_ascii=False)

    html = HTML_TEMPLATE.replace('__DATA_JSON__', data_json).replace('__SPECS_JSON__', specs_json)

    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(html)

    print(f'✅ Dashboard généré : {Path(output_path).resolve()}')
    print(f'   {len(records)} entrées · {df["parser"].nunique()} parseur(s)')


# ─── Template HTML (données injectées via __DATA_JSON__ et __SPECS_JSON__) ─
HTML_TEMPLATE = r"""
<!DOCTYPE html>
<html lang="fr">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>OCR Benchmark Dashboard</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.js"></script>
<style>
@import url('https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=DM+Sans:wght@300;400;500;600&display=swap');
*, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }
:root {
  --bg: #0d0f14; --surface: #141720; --surface2: #1c2030;
  --border: rgba(255,255,255,0.07); --border2: rgba(255,255,255,0.14);
  --text: #e8eaf0; --muted: #6b7080;
  --accent: #4f9cf9; --accent2: #7b6ef6;
  --success: #4ade80; --warning: #fbbf24; --danger: #f87171; --teal: #2dd4bf;
  --mono: 'Space Mono', monospace; --sans: 'DM Sans', sans-serif;
}
body { background: var(--bg); color: var(--text); font-family: var(--sans); font-size: 14px; min-height: 100vh; }
.header { border-bottom:1px solid var(--border); padding:20px 32px; display:flex; align-items:center; justify-content:space-between; background:var(--surface); position:sticky; top:0; z-index:10; }
.header h1 { font-family:var(--mono); font-size:17px; font-weight:700; }
.header p { color:var(--muted); font-size:11px; margin-top:3px; font-family:var(--mono); }
.badge { background:rgba(79,156,249,0.12); color:var(--accent); border:1px solid rgba(79,156,249,0.25); padding:4px 10px; border-radius:4px; font-family:var(--mono); font-size:11px; }
.main { max-width:1400px; margin:0 auto; padding:28px 32px; }
.filters-bar { display:flex; gap:12px; margin-bottom:24px; flex-wrap:wrap; align-items:center; background:var(--surface); border:1px solid var(--border); border-radius:10px; padding:14px 18px; }
.filter-label { font-size:11px; font-family:var(--mono); color:var(--muted); text-transform:uppercase; letter-spacing:.5px; }
.parser-toggle { padding:5px 12px; border-radius:20px; border:1px solid var(--border2); background:transparent; color:var(--muted); font-size:12px; font-family:var(--mono); cursor:pointer; transition:all .15s; }
.parser-toggle.active { color:#0d0f14; border-color:transparent; font-weight:700; }
select { background:var(--surface); border:1px solid var(--border2); color:var(--text); padding:7px 12px; border-radius:6px; font-family:var(--sans); font-size:13px; cursor:pointer; outline:none; }
select:focus { border-color:var(--accent); }
.btn { background:var(--accent); color:#0d0f14; border:none; padding:8px 16px; border-radius:6px; font-family:var(--sans); font-size:13px; font-weight:600; cursor:pointer; transition:opacity .15s; }
.btn:hover { opacity:.85; }
.btn-ghost { background:transparent; color:var(--text); border:1px solid var(--border2); }
.btn-ghost:hover { background:var(--surface2); }
.kpi-row { display:grid; grid-template-columns:repeat(auto-fit,minmax(200px,1fr)); gap:16px; margin-bottom:32px; }
.kpi-card { background:var(--surface); border:1px solid var(--border); border-radius:10px; padding:20px; position:relative; overflow:hidden; }
.kpi-card::before { content:''; position:absolute; top:0; left:0; right:0; height:2px; }
.kpi-card.c-blue::before { background:var(--accent); } .kpi-card.c-green::before { background:var(--success); }
.kpi-card.c-purple::before { background:var(--accent2); } .kpi-card.c-teal::before { background:var(--teal); }
.kpi-card .label { font-size:11px; font-family:var(--mono); color:var(--muted); text-transform:uppercase; letter-spacing:.5px; margin-bottom:10px; }
.kpi-card .value { font-size:28px; font-weight:600; font-family:var(--mono); line-height:1; }
.kpi-card .sub { font-size:11px; color:var(--muted); margin-top:6px; font-family:var(--mono); }
.kpi-card.c-blue .value{color:var(--accent)} .kpi-card.c-green .value{color:var(--success)} .kpi-card.c-purple .value{color:var(--accent2)} .kpi-card.c-teal .value{color:var(--teal)}
.section-title { font-family:var(--mono); font-size:12px; text-transform:uppercase; letter-spacing:1px; color:var(--muted); margin-bottom:16px; display:flex; align-items:center; gap:10px; }
.section-title::after { content:''; flex:1; height:1px; background:var(--border); }
.chart-grid { display:grid; grid-template-columns:1fr 1fr; gap:20px; margin-bottom:32px; }
@media(max-width:900px){.chart-grid{grid-template-columns:1fr;}}
.chart-card { background:var(--surface); border:1px solid var(--border); border-radius:10px; padding:20px; }
.chart-card h3 { font-size:13px; font-weight:500; font-family:var(--mono); margin-bottom:4px; }
.chart-card .chart-sub { font-size:11px; color:var(--muted); margin-bottom:12px; font-family:var(--mono); }
.chart-full { background:var(--surface); border:1px solid var(--border); border-radius:10px; padding:20px; margin-bottom:32px; }
.chart-full h3 { font-size:13px; font-weight:500; font-family:var(--mono); margin-bottom:4px; }
.chart-full .chart-sub { font-size:11px; color:var(--muted); margin-bottom:12px; font-family:var(--mono); }
.legend { display:flex; flex-wrap:wrap; gap:14px; margin-bottom:10px; }
.legend-item { display:flex; align-items:center; gap:6px; font-size:12px; font-family:var(--mono); color:var(--muted); }
.legend-dot { width:10px; height:10px; border-radius:2px; }
.table-wrap { background:var(--surface); border:1px solid var(--border); border-radius:10px; overflow:hidden; margin-bottom:32px; }
.table-header { padding:16px 20px; border-bottom:1px solid var(--border); display:flex; align-items:center; justify-content:space-between; }
.table-header h3 { font-family:var(--mono); font-size:13px; font-weight:500; }
table { width:100%; border-collapse:collapse; }
thead th { padding:10px 16px; text-align:left; font-family:var(--mono); font-size:11px; color:var(--muted); text-transform:uppercase; letter-spacing:.5px; background:var(--surface2); cursor:pointer; user-select:none; white-space:nowrap; }
thead th:hover { color:var(--text); }
tbody tr { border-top:1px solid var(--border); transition:background .1s; }
tbody tr:hover { background:var(--surface2); }
tbody td { padding:11px 16px; font-size:13px; white-space:nowrap; }
.td-model { font-family:var(--mono); font-weight:700; }
.td-parser { font-family:var(--mono); font-size:11px; padding:3px 8px; border-radius:4px; }
.bar-cell { display:flex; align-items:center; gap:10px; }
.bar-track { flex:1; max-width:90px; height:4px; background:var(--surface2); border-radius:2px; overflow:hidden; }
.bar-fill { height:100%; border-radius:2px; }
.td-val { font-family:var(--mono); font-size:13px; min-width:45px; }
.rank-badge { display:inline-flex; align-items:center; justify-content:center; width:22px; height:22px; border-radius:4px; font-family:var(--mono); font-size:11px; font-weight:700; }
.rank-1{background:rgba(251,191,36,.15);color:var(--warning);border:1px solid rgba(251,191,36,.3)}
.rank-2{background:rgba(148,163,184,.1);color:#94a3b8;border:1px solid rgba(148,163,184,.2)}
.rank-3{background:rgba(180,130,80,.1);color:#b48250;border:1px solid rgba(180,130,80,.2)}
.rank-other{background:var(--surface2);color:var(--muted);border:1px solid var(--border)}

/* ── FOOTER ── */
.footer { border-top:1px solid var(--border); background:var(--surface); margin-top:16px; }
.footer-toggle { width:100%; display:flex; align-items:center; justify-content:space-between; padding:16px 32px; background:transparent; border:none; color:var(--muted); font-family:var(--mono); font-size:12px; text-transform:uppercase; letter-spacing:1px; cursor:pointer; transition:color .15s; }
.footer-toggle:hover { color:var(--text); }
.footer-toggle .arrow { transition:transform .25s; font-size:14px; }
.footer-toggle.open .arrow { transform:rotate(180deg); }
.footer-body { display:none; padding:0 32px 32px; max-width:1400px; margin:0 auto; }
.footer-body.open { display:block; }
.specs-grid { display:grid; grid-template-columns:repeat(auto-fit,minmax(280px,1fr)); gap:20px; margin-top:8px; }
.spec-card { background:var(--surface2); border:1px solid var(--border); border-radius:10px; padding:20px; }
.spec-card h4 { font-family:var(--mono); font-size:11px; text-transform:uppercase; letter-spacing:1px; color:var(--accent); margin-bottom:14px; display:flex; align-items:center; gap:8px; }
.spec-row { display:flex; justify-content:space-between; align-items:baseline; padding:5px 0; border-bottom:1px solid var(--border); gap:12px; }
.spec-row:last-child { border-bottom:none; }
.spec-key { font-size:12px; color:var(--muted); font-family:var(--mono); white-space:nowrap; }
.spec-val { font-size:12px; color:var(--text); font-family:var(--mono); text-align:right; word-break:break-all; }
.spec-val.accent { color:var(--accent); }
.spec-val.success { color:var(--success); }
.spec-val.teal { color:var(--teal); }
.spec-val.warning { color:var(--warning); }
.pkg-grid { display:grid; grid-template-columns:1fr 1fr; gap:4px 16px; }
.pkg-row { display:flex; justify-content:space-between; padding:4px 0; border-bottom:1px solid var(--border); }
.pkg-row:last-child { border-bottom:none; }
.pkg-name { font-family:var(--mono); font-size:12px; color:var(--teal); }
.pkg-ver  { font-family:var(--mono); font-size:12px; color:var(--muted); }
.model-row { display:flex; flex-direction:column; padding:6px 0; border-bottom:1px solid var(--border); gap:2px; }
.model-row:last-child { border-bottom:none; }
.model-key { font-family:var(--mono); font-size:12px; font-weight:700; color:var(--text); }
.model-desc { font-family:var(--mono); font-size:11px; color:var(--muted); }
.notes-box { background:var(--bg); border:1px solid var(--border); border-radius:6px; padding:12px 16px; font-family:var(--mono); font-size:12px; color:var(--muted); line-height:1.7; margin-top:8px; }
</style>
</head>
<body>
<div class="header">
  <div>
    <h1>OCR_BENCHMARK_DASHBOARD</h1>
    <p id="headerSub">Chargement...</p>
  </div>
  <span class="badge" id="headerBadge"></span>
</div>
<div class="main">
  <div class="filters-bar">
    <span class="filter-label">Parseur</span>
    <div id="parserToggles" style="display:flex;gap:6px;flex-wrap:wrap"></div>
    <span class="filter-label" style="margin-left:12px">Modèle OCR</span>
    <select id="ocrFilter"><option value="all">Tous</option></select>
    <span class="filter-label" style="margin-left:12px">Trier tableau par</span>
    <select id="sortBy">
      <option value="gt_accuracy_pct">GT Accuracy</option>
      <option value="fill_rate_pct">Fill Rate</option>
      <option value="llm_judge_score_pct">LLM Judge</option>
      <option value="avg_latency_s">Latence (ASC)</option>
    </select>
    <button class="btn btn-ghost" onclick="resetFilters()" style="margin-left:auto">↺ Reset</button>
  </div>
  <div id="kpiRow" class="kpi-row"></div>
  <div class="section-title">Précision &amp; Qualité</div>
  <div class="chart-grid">
    <div class="chart-card"><h3>GT Accuracy par modèle OCR</h3><p class="chart-sub">% ground-truth · par parseur</p><div id="legAccuracy" class="legend"></div><div style="position:relative;height:260px"><canvas id="chartAccuracy" role="img" aria-label="GT Accuracy">GT Accuracy</canvas></div></div>
    <div class="chart-card"><h3>Fill Rate par modèle OCR</h3><p class="chart-sub">% champs remplis · par parseur</p><div id="legFill" class="legend"></div><div style="position:relative;height:260px"><canvas id="chartFill" role="img" aria-label="Fill Rate">Fill Rate</canvas></div></div>
  </div>
  <div class="section-title">Performance &amp; Latence</div>
  <div class="chart-grid">
    <div class="chart-card"><h3>Latence moyenne (secondes)</h3><p class="chart-sub">avg_latency_s · plus bas = meilleur</p><div id="legLatency" class="legend"></div><div style="position:relative;height:260px"><canvas id="chartLatency" role="img" aria-label="Latence">Latence</canvas></div></div>
    <div class="chart-card"><h3>LLM Judge Score</h3><p class="chart-sub">Score évaluation LLM juge · par parseur</p><div id="legJudge" class="legend"></div><div style="position:relative;height:260px"><canvas id="chartJudge" role="img" aria-label="LLM Judge">LLM Judge</canvas></div></div>
  </div>
  <div class="section-title">Vue scatter — Accuracy vs Latence</div>
  <div class="chart-full">
    <h3>Accuracy vs Latence</h3>
    <p class="chart-sub">X = latence (s) · Y = GT Accuracy (%) · taille bulle = fill rate · idéal = haut-gauche</p>
    <div id="legScatter" class="legend"></div>
    <div style="position:relative;height:320px"><canvas id="chartScatter" role="img" aria-label="Scatter">Scatter</canvas></div>
  </div>
  <div class="section-title">Données détaillées</div>
  <div class="table-wrap">
    <div class="table-header"><h3>Tous les résultats</h3><span id="rowCount" style="font-size:12px;font-family:var(--mono);color:var(--muted)"></span></div>
    <div style="overflow-x:auto">
      <table><thead><tr>
        <th>#</th>
        <th onclick="sortTable('model')">Modèle ↕</th>
        <th onclick="sortTable('parser')">Parseur ↕</th>
        <th onclick="sortTable('gt_accuracy_pct')">GT Accuracy ↕</th>
        <th onclick="sortTable('fill_rate_pct')">Fill Rate ↕</th>
        <th onclick="sortTable('llm_judge_score_pct')">LLM Judge ↕</th>
        <th onclick="sortTable('avg_latency_s')">Latence (s) ↕</th>
      </tr></thead>
      <tbody id="tableBody"></tbody></table>
    </div>
  </div>
</div>

<!-- ── FOOTER SPECS ── -->
<footer class="footer">
  <button class="footer-toggle" id="footerToggle" onclick="toggleFooter()">
    <span>⚙ Spécifications techniques du benchmark</span>
    <span class="arrow">▾</span>
  </button>
  <div class="footer-body" id="footerBody"></div>
</footer>

<script>
const RAW_DATA = __DATA_JSON__;
const SPECS    = __SPECS_JSON__;
const COLORS = ['#4f9cf9','#7b6ef6','#2dd4bf','#fbbf24','#f87171','#4ade80','#fb923c','#e879f9'];
let parsers = [...new Set(RAW_DATA.map(r=>r.parser))];
let activeParsers = new Set(parsers);
let ocrFilter = 'all';
let tableSortCol = 'gt_accuracy_pct', tableSortDir = -1;
const charts = {};

function col(p,a=1){const h=COLORS[parsers.indexOf(p)%COLORS.length];if(a===1)return h;const r=parseInt(h.slice(1,3),16),g=parseInt(h.slice(3,5),16),b=parseInt(h.slice(5,7),16);return `rgba(${r},${g},${b},${a})`;}
function fmt(v){return v==null?'—':typeof v==='number'?v.toFixed(1):v;}
function filtered(){return RAW_DATA.filter(r=>activeParsers.has(r.parser)&&(ocrFilter==='all'||r.model===ocrFilter));}

function buildToggles(){
  document.getElementById('parserToggles').innerHTML=parsers.map((p,i)=>`
    <button class="parser-toggle active" data-i="${i}" onclick="toggleP('${p}',this,${i})"
      style="background:${COLORS[i]};border-color:${COLORS[i]};color:#0d0f14">${p}</button>`).join('');
}
function toggleP(p,btn,i){
  if(activeParsers.has(p)){if(activeParsers.size===1)return;activeParsers.delete(p);btn.classList.remove('active');btn.style.cssText=`background:transparent;color:var(--muted);border-color:var(--border2)`;}
  else{activeParsers.add(p);btn.classList.add('active');btn.style.cssText=`background:${COLORS[i]};color:#0d0f14;border-color:${COLORS[i]}`;}
  updateAll();
}
function buildOcrFilter(){
  const models=[...new Set(RAW_DATA.map(r=>r.model))].sort();
  const sel=document.getElementById('ocrFilter');
  sel.innerHTML='<option value="all">Tous</option>'+models.map(m=>`<option value="${m}">${m}</option>`).join('');
  sel.onchange=()=>{ocrFilter=sel.value;updateAll();};
}
function resetFilters(){
  activeParsers=new Set(parsers);ocrFilter='all';
  document.getElementById('ocrFilter').value='all';
  document.querySelectorAll('.parser-toggle').forEach((b,i)=>{b.classList.add('active');b.style.cssText=`background:${COLORS[i]};color:#0d0f14;border-color:${COLORS[i]}`;});
  updateAll();
}
document.getElementById('sortBy').onchange=function(){tableSortCol=this.value;tableSortDir=this.value==='avg_latency_s'?1:-1;buildTable(filtered());};

function buildKPIs(data){
  if(!data.length)return;
  const best=data.reduce((a,b)=>(b.gt_accuracy_pct||0)>(a.gt_accuracy_pct||0)?b:a);
  const avg=k=>(data.reduce((s,r)=>s+(r[k]||0),0)/data.length).toFixed(1);
  document.getElementById('kpiRow').innerHTML=`
    <div class="kpi-card c-green"><div class="label">Meilleur GT Accuracy</div><div class="value">${fmt(best.gt_accuracy_pct)}%</div><div class="sub">${best.model} · ${best.parser}</div></div>
    <div class="kpi-card c-blue"><div class="label">Accuracy moyenne</div><div class="value">${avg('gt_accuracy_pct')}%</div><div class="sub">${data.length} entrées filtrées</div></div>
    <div class="kpi-card c-teal"><div class="label">Fill Rate moyen</div><div class="value">${avg('fill_rate_pct')}%</div><div class="sub">champs remplis</div></div>
    <div class="kpi-card c-purple"><div class="label">Latence moyenne</div><div class="value">${avg('avg_latency_s')}s</div><div class="sub">avg_latency_s</div></div>`;
}

function legend(id,pArr){document.getElementById(id).innerHTML=pArr.map(p=>`<span class="legend-item"><span class="legend-dot" style="background:${col(p)}"></span>${p}</span>`).join('');}

const tick={color:'#6b7080',font:{size:11,family:"'Space Mono',monospace"}};
const grid={color:'rgba(255,255,255,0.05)'};
const baseOpts={responsive:true,maintainAspectRatio:false,plugins:{legend:{display:false},tooltip:{bodyFont:{family:"'Space Mono',monospace",size:12}}},scales:{x:{ticks:{...tick,maxRotation:35},grid},y:{ticks:tick,grid}}};

function buildCharts(data){
  const activePArr=[...activeParsers].filter(p=>data.some(r=>r.parser===p));
  const models=[...new Set(data.map(r=>r.model))].sort();
  function ds(metric){return activePArr.map(p=>({label:p,data:models.map(m=>{const r=data.find(d=>d.model===m&&d.parser===p);return r?(r[metric]??null):null;}),backgroundColor:col(p,.75),borderColor:col(p),borderWidth:1.5,borderRadius:3}));}
  ['chartAccuracy','chartFill','chartLatency','chartJudge','chartScatter'].forEach(id=>{if(charts[id]){charts[id].destroy();delete charts[id];}});
  charts.chartAccuracy=new Chart(document.getElementById('chartAccuracy'),{type:'bar',data:{labels:models,datasets:ds('gt_accuracy_pct')},options:{...baseOpts,scales:{...baseOpts.scales,y:{...baseOpts.scales.y,max:100}}}});
  legend('legAccuracy',activePArr);
  charts.chartFill=new Chart(document.getElementById('chartFill'),{type:'bar',data:{labels:models,datasets:ds('fill_rate_pct')},options:{...baseOpts,scales:{...baseOpts.scales,y:{...baseOpts.scales.y,max:100}}}});
  legend('legFill',activePArr);
  charts.chartLatency=new Chart(document.getElementById('chartLatency'),{type:'bar',data:{labels:models,datasets:ds('avg_latency_s')},options:baseOpts});
  legend('legLatency',activePArr);
  charts.chartJudge=new Chart(document.getElementById('chartJudge'),{type:'bar',data:{labels:models,datasets:ds('llm_judge_score_pct')},options:{...baseOpts,scales:{...baseOpts.scales,y:{...baseOpts.scales.y,max:100}}}});
  legend('legJudge',activePArr);
  charts.chartScatter=new Chart(document.getElementById('chartScatter'),{type:'bubble',
    data:{datasets:activePArr.map(p=>({label:p,data:data.filter(r=>r.parser===p).map(r=>({x:r.avg_latency_s,y:r.gt_accuracy_pct,r:Math.max(5,(r.fill_rate_pct/100)*18),model:r.model,fill:r.fill_rate_pct})),backgroundColor:col(p,.5),borderColor:col(p),borderWidth:1.5}))},
    options:{responsive:true,maintainAspectRatio:false,plugins:{legend:{display:false},tooltip:{callbacks:{label:c=>`${c.raw.model} (${c.dataset.label}) acc:${c.raw.y}% lat:${c.raw.x}s fill:${c.raw.fill}%`},bodyFont:{family:"'Space Mono',monospace",size:11}}},
    scales:{x:{title:{display:true,text:'Latence (s)',color:'#6b7080'},ticks:tick,grid},y:{title:{display:true,text:'GT Accuracy (%)',color:'#6b7080'},ticks:tick,grid,min:0,max:105}}}});
  legend('legScatter',activePArr);
}

function sortTable(c){if(tableSortCol===c)tableSortDir*=-1;else{tableSortCol=c;tableSortDir=-1;}buildTable(filtered());}
function buildTable(data){
  const sorted=[...data].sort((a,b)=>{const av=a[tableSortCol]??'',bv=b[tableSortCol]??'';return(typeof av==='number'?(av-bv):String(av).localeCompare(String(bv)))*tableSortDir;});
  document.getElementById('rowCount').textContent=`${sorted.length} lignes`;
  const maxV=k=>Math.max(...data.map(r=>r[k]||0));
  const [mA,mF,mJ,mL]=['gt_accuracy_pct','fill_rate_pct','llm_judge_score_pct','avg_latency_s'].map(maxV);
  function bar(v,max,c){const p=max?(v/max*100):0;return `<div class="bar-cell"><span class="td-val">${fmt(v)}</span><div class="bar-track"><div class="bar-fill" style="width:${p}%;background:${c}"></div></div></div>`;}
  document.getElementById('tableBody').innerHTML=sorted.map((r,i)=>{
    const pi=parsers.indexOf(r.parser),c=COLORS[pi%COLORS.length],rc=i<3?`rank-${i+1}`:'rank-other';
    return `<tr><td><span class="rank-badge ${rc}">${i+1}</span></td><td class="td-model">${r.model}</td><td><span class="td-parser" style="background:${c}22;color:${c};border:1px solid ${c}44">${r.parser}</span></td><td>${bar(r.gt_accuracy_pct,mA,'#4ade80')}</td><td>${bar(r.fill_rate_pct,mF,'#4f9cf9')}</td><td>${bar(r.llm_judge_score_pct,mJ,'#7b6ef6')}</td><td>${bar(r.avg_latency_s,mL,'#fbbf24')}</td></tr>`;
  }).join('');
}
function updateAll(){const d=filtered();buildKPIs(d);buildCharts(d);buildTable(d);}

// ── FOOTER ──────────────────────────────────────────────────────────────
function toggleFooter(){
  const btn=document.getElementById('footerToggle');
  const body=document.getElementById('footerBody');
  btn.classList.toggle('open');
  body.classList.toggle('open');
}

function buildFooter(){
  const s = SPECS;
  if(!s || !Object.keys(s).length) return;

  function row(k,v,cls=''){return `<div class="spec-row"><span class="spec-key">${k}</span><span class="spec-val ${cls}">${v||'—'}</span></div>`;}

  // Card 1 — Environnement
  const envFields = [
    ['date',s.date,'accent'],['os',s.os,''],['python',s.python_version,'teal'],
    ['cpu',s.cpu,''],['gpu',s.gpu,'warning'],['ram',s.ram,''],
    ['pipeline',s.pipeline_version,'accent']
  ].filter(([,v])=>v);

  // Card 2 — Dataset & évaluation
  const evalFields = [
    ['dataset',s.dataset,''],['split',s.dataset_split,''],
    ['métriques',s.eval_metric,'teal'],['runs',s.n_runs,'']
  ].filter(([,v])=>v);

  // Card 3 — Packages
  const pkgs = s.packages ? Object.entries(s.packages) : [];

  // Card 4 — Modèles OCR
  const ocrModels = s.ocr_models ? Object.entries(s.ocr_models) : [];

  // Card 5 — Parseurs LLM
  const llmParsers = s.llm_parsers ? Object.entries(s.llm_parsers) : [];

  let html = '<div class="specs-grid">';

  // Env card
  if(envFields.length){
    html += `<div class="spec-card"><h4>🖥 Environnement</h4>${envFields.map(([k,v,c])=>row(k,v,c)).join('')}</div>`;
  }

  // Eval card
  if(evalFields.length){
    html += `<div class="spec-card"><h4>📐 Dataset &amp; Évaluation</h4>${evalFields.map(([k,v,c])=>row(k,v,c)).join('')}</div>`;
  }

  // Packages card
  if(pkgs.length){
    const pkgRows = pkgs.map(([n,v])=>`<div class="pkg-row"><span class="pkg-name">${n}</span><span class="pkg-ver">${v}</span></div>`).join('');
    html += `<div class="spec-card"><h4>📦 Packages Python</h4><div class="pkg-grid">${pkgRows}</div></div>`;
  }

  // OCR models card
  if(ocrModels.length){
    const mRows = ocrModels.map(([k,v])=>`<div class="model-row"><span class="model-key">${k}</span><span class="model-desc">${v}</span></div>`).join('');
    html += `<div class="spec-card"><h4>🔍 Modèles OCR testés</h4>${mRows}</div>`;
  }

  // LLM parsers card
  if(llmParsers.length){
    const pRows = llmParsers.map(([k,v])=>`<div class="model-row"><span class="model-key">${k}</span><span class="model-desc">${v}</span></div>`).join('');
    html += `<div class="spec-card"><h4>🤖 Parseurs LLM testés</h4>${pRows}</div>`;
  }

  html += '</div>';

  // Notes
  if(s.notes){
    html += `<div class="notes-box">📝 &nbsp;${s.notes}</div>`;
  }

  document.getElementById('footerBody').innerHTML = html;
}

// Init
buildToggles();
buildOcrFilter();
document.getElementById('headerSub').textContent=`// ${parsers.length} parseur(s) · ${RAW_DATA.length} entrées`;
document.getElementById('headerBadge').textContent=`${parsers.length} parseurs chargés`;
updateAll();
buildFooter();
</script>
</body></html>
"""

generate_dashboard(df, OUTPUT_HTML, BENCHMARK_SPECS)

In [ ]:
def generate_dashboard(df: pd.DataFrame, output_path: str, specs: dict = None):
    records = df.where(pd.notnull(df), None).to_dict(orient='records')
    data_json  = json.dumps(records, ensure_ascii=False)
    specs_json = json.dumps(specs or {}, ensure_ascii=False)

    html = HTML_TEMPLATE.replace('__DATA_JSON__', data_json).replace('__SPECS_JSON__', specs_json)

    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(html)

    print(f'✅ Dashboard généré : {Path(output_path).resolve()}')
    print(f'   {len(records)} entrées · {df["parser"].nunique()} parseur(s)')


HTML_TEMPLATE = r"""
<!DOCTYPE html>
<html lang="fr">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>OCR Benchmark Dashboard</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.js"></script>
<style>
@import url('https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=DM+Sans:wght@300;400;500;600&display=swap');
*, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }
:root {
  --bg: #0d0f14; --surface: #141720; --surface2: #1c2030;
  --border: rgba(255,255,255,0.07); --border2: rgba(255,255,255,0.14);
  --text: #e8eaf0; --muted: #6b7080;
  --accent: #4f9cf9; --accent2: #7b6ef6;
  --success: #4ade80; --warning: #fbbf24; --danger: #f87171; --teal: #2dd4bf;
  --mono: 'Space Mono', monospace; --sans: 'DM Sans', sans-serif;
}
body { background: var(--bg); color: var(--text); font-family: var(--sans); font-size: 14px; min-height: 100vh; }
.header { border-bottom:1px solid var(--border); padding:20px 32px; display:flex; align-items:center; justify-content:space-between; background:var(--surface); position:sticky; top:0; z-index:10; }
.header h1 { font-family:var(--mono); font-size:17px; font-weight:700; }
.header p { color:var(--muted); font-size:11px; margin-top:3px; font-family:var(--mono); }
.badge { background:rgba(79,156,249,0.12); color:var(--accent); border:1px solid rgba(79,156,249,0.25); padding:4px 10px; border-radius:4px; font-family:var(--mono); font-size:11px; }
.main { max-width:1400px; margin:0 auto; padding:28px 32px; }
.filters-bar { display:flex; gap:12px; margin-bottom:24px; flex-wrap:wrap; align-items:center; background:var(--surface); border:1px solid var(--border); border-radius:10px; padding:14px 18px; }
.filter-label { font-size:11px; font-family:var(--mono); color:var(--muted); text-transform:uppercase; letter-spacing:.5px; }
.parser-toggle { padding:5px 12px; border-radius:20px; border:1px solid var(--border2); background:transparent; color:var(--muted); font-size:12px; font-family:var(--mono); cursor:pointer; transition:all .15s; }
.parser-toggle.active { color:#0d0f14; border-color:transparent; font-weight:700; }
select { background:var(--surface); border:1px solid var(--border2); color:var(--text); padding:7px 12px; border-radius:6px; font-family:var(--sans); font-size:13px; cursor:pointer; outline:none; }
select:focus { border-color:var(--accent); }
.btn { background:var(--accent); color:#0d0f14; border:none; padding:8px 16px; border-radius:6px; font-family:var(--sans); font-size:13px; font-weight:600; cursor:pointer; transition:opacity .15s; }
.btn:hover { opacity:.85; }
.btn-ghost { background:transparent; color:var(--text); border:1px solid var(--border2); }
.btn-ghost:hover { background:var(--surface2); }
.kpi-row { display:grid; grid-template-columns:repeat(auto-fit,minmax(200px,1fr)); gap:16px; margin-bottom:32px; }
.kpi-card { background:var(--surface); border:1px solid var(--border); border-radius:10px; padding:20px; position:relative; overflow:hidden; }
.kpi-card::before { content:''; position:absolute; top:0; left:0; right:0; height:2px; }
.kpi-card.c-blue::before { background:var(--accent); } .kpi-card.c-green::before { background:var(--success); }
.kpi-card.c-purple::before { background:var(--accent2); } .kpi-card.c-teal::before { background:var(--teal); }
.kpi-card.c-red::before { background:var(--danger); }
.kpi-card .label { font-size:11px; font-family:var(--mono); color:var(--muted); text-transform:uppercase; letter-spacing:.5px; margin-bottom:10px; }
.kpi-card .value { font-size:28px; font-weight:600; font-family:var(--mono); line-height:1; }
.kpi-card .sub { font-size:11px; color:var(--muted); margin-top:6px; font-family:var(--mono); }
.kpi-card.c-blue .value{color:var(--accent)} .kpi-card.c-green .value{color:var(--success)} .kpi-card.c-purple .value{color:var(--accent2)} .kpi-card.c-teal .value{color:var(--teal)} .kpi-card.c-red .value{color:var(--danger)}
.section-title { font-family:var(--mono); font-size:12px; text-transform:uppercase; letter-spacing:1px; color:var(--muted); margin-bottom:16px; display:flex; align-items:center; gap:10px; }
.section-title::after { content:''; flex:1; height:1px; background:var(--border); }
.chart-grid { display:grid; grid-template-columns:1fr 1fr; gap:20px; margin-bottom:32px; }
@media(max-width:900px){.chart-grid{grid-template-columns:1fr;}}
.chart-card { background:var(--surface); border:1px solid var(--border); border-radius:10px; padding:20px; }
.chart-card h3 { font-size:13px; font-weight:500; font-family:var(--mono); margin-bottom:4px; }
.chart-card .chart-sub { font-size:11px; color:var(--muted); margin-bottom:12px; font-family:var(--mono); }
.chart-full { background:var(--surface); border:1px solid var(--border); border-radius:10px; padding:20px; margin-bottom:32px; }
.chart-full h3 { font-size:13px; font-weight:500; font-family:var(--mono); margin-bottom:4px; }
.chart-full .chart-sub { font-size:11px; color:var(--muted); margin-bottom:12px; font-family:var(--mono); }
.legend { display:flex; flex-wrap:wrap; gap:14px; margin-bottom:10px; }
.legend-item { display:flex; align-items:center; gap:6px; font-size:12px; font-family:var(--mono); color:var(--muted); }
.legend-dot { width:10px; height:10px; border-radius:2px; }
.table-wrap { background:var(--surface); border:1px solid var(--border); border-radius:10px; overflow:hidden; margin-bottom:32px; }
.table-header { padding:16px 20px; border-bottom:1px solid var(--border); display:flex; align-items:center; justify-content:space-between; }
.table-header h3 { font-family:var(--mono); font-size:13px; font-weight:500; }
table { width:100%; border-collapse:collapse; }
thead th { padding:10px 16px; text-align:left; font-family:var(--mono); font-size:11px; color:var(--muted); text-transform:uppercase; letter-spacing:.5px; background:var(--surface2); cursor:pointer; user-select:none; white-space:nowrap; }
thead th:hover { color:var(--text); }
tbody tr { border-top:1px solid var(--border); transition:background .1s; }
tbody tr:hover { background:var(--surface2); }
tbody td { padding:11px 16px; font-size:13px; white-space:nowrap; }
.td-model { font-family:var(--mono); font-weight:700; }
.td-parser { font-family:var(--mono); font-size:11px; padding:3px 8px; border-radius:4px; }
.bar-cell { display:flex; align-items:center; gap:10px; }
.bar-track { flex:1; max-width:90px; height:4px; background:var(--surface2); border-radius:2px; overflow:hidden; }
.bar-fill { height:100%; border-radius:2px; }
.td-val { font-family:var(--mono); font-size:13px; min-width:45px; }
.rank-badge { display:inline-flex; align-items:center; justify-content:center; width:22px; height:22px; border-radius:4px; font-family:var(--mono); font-size:11px; font-weight:700; }
.rank-1{background:rgba(251,191,36,.15);color:var(--warning);border:1px solid rgba(251,191,36,.3)}
.rank-2{background:rgba(148,163,184,.1);color:#94a3b8;border:1px solid rgba(148,163,184,.2)}
.rank-3{background:rgba(180,130,80,.1);color:#b48250;border:1px solid rgba(180,130,80,.2)}
.rank-other{background:var(--surface2);color:var(--muted);border:1px solid var(--border)}
.footer { border-top:1px solid var(--border); background:var(--surface); margin-top:16px; }
.footer-toggle { width:100%; display:flex; align-items:center; justify-content:space-between; padding:16px 32px; background:transparent; border:none; color:var(--muted); font-family:var(--mono); font-size:12px; text-transform:uppercase; letter-spacing:1px; cursor:pointer; transition:color .15s; }
.footer-toggle:hover { color:var(--text); }
.footer-toggle .arrow { transition:transform .25s; font-size:14px; }
.footer-toggle.open .arrow { transform:rotate(180deg); }
.footer-body { display:none; padding:0 32px 32px; max-width:1400px; margin:0 auto; }
.footer-body.open { display:block; }
.specs-grid { display:grid; grid-template-columns:repeat(auto-fit,minmax(280px,1fr)); gap:20px; margin-top:8px; }
.spec-card { background:var(--surface2); border:1px solid var(--border); border-radius:10px; padding:20px; }
.spec-card h4 { font-family:var(--mono); font-size:11px; text-transform:uppercase; letter-spacing:1px; color:var(--accent); margin-bottom:14px; display:flex; align-items:center; gap:8px; }
.spec-row { display:flex; justify-content:space-between; align-items:baseline; padding:5px 0; border-bottom:1px solid var(--border); gap:12px; }
.spec-row:last-child { border-bottom:none; }
.spec-key { font-size:12px; color:var(--muted); font-family:var(--mono); white-space:nowrap; }
.spec-val { font-size:12px; color:var(--text); font-family:var(--mono); text-align:right; word-break:break-all; }
.spec-val.accent { color:var(--accent); }
.spec-val.success { color:var(--success); }
.spec-val.teal { color:var(--teal); }
.spec-val.warning { color:var(--warning); }
.pkg-grid { display:grid; grid-template-columns:1fr 1fr; gap:4px 16px; }
.pkg-row { display:flex; justify-content:space-between; padding:4px 0; border-bottom:1px solid var(--border); }
.pkg-row:last-child { border-bottom:none; }
.pkg-name { font-family:var(--mono); font-size:12px; color:var(--teal); }
.pkg-ver  { font-family:var(--mono); font-size:12px; color:var(--muted); }
.model-row { display:flex; flex-direction:column; padding:6px 0; border-bottom:1px solid var(--border); gap:2px; }
.model-row:last-child { border-bottom:none; }
.model-key { font-family:var(--mono); font-size:12px; font-weight:700; color:var(--text); }
.model-desc { font-family:var(--mono); font-size:11px; color:var(--muted); }
.notes-box { background:var(--bg); border:1px solid var(--border); border-radius:6px; padding:12px 16px; font-family:var(--mono); font-size:12px; color:var(--muted); line-height:1.7; margin-top:8px; }
</style>
</head>
<body>
<div class="header">
  <div>
    <h1>OCR_BENCHMARK_DASHBOARD</h1>
    <p id="headerSub">Chargement...</p>
  </div>
  <span class="badge" id="headerBadge"></span>
</div>
<div class="main">
  <div class="filters-bar">
    <span class="filter-label">Parseur</span>
    <div id="parserToggles" style="display:flex;gap:6px;flex-wrap:wrap"></div>
    <span class="filter-label" style="margin-left:12px">Modèle OCR</span>
    <select id="ocrFilter"><option value="all">Tous</option></select>
    <span class="filter-label" style="margin-left:12px">Trier tableau par</span>
    <select id="sortBy">
      <option value="gt_accuracy_pct">GT Accuracy</option>
      <option value="fill_rate_pct">Fill Rate</option>
      <option value="avg_latency_s">Latence (ASC)</option>
    </select>
    <button class="btn btn-ghost" onclick="resetFilters()" style="margin-left:auto">↺ Reset</button>
  </div>
  <div id="kpiRow" class="kpi-row"></div>
  <div class="section-title">Précision &amp; Qualité</div>
  <div class="chart-grid">
    <div class="chart-card"><h3>GT Accuracy par modèle OCR</h3><p class="chart-sub">% ground-truth · par parseur</p><div id="legAccuracy" class="legend"></div><div style="position:relative;height:260px"><canvas id="chartAccuracy"></canvas></div></div>
    <div class="chart-card"><h3>Fill Rate par modèle OCR</h3><p class="chart-sub">% champs remplis · par parseur</p><div id="legFill" class="legend"></div><div style="position:relative;height:260px"><canvas id="chartFill"></canvas></div></div>
  </div>
  <div class="section-title">Performance &amp; Latence</div>
  <div class="chart-grid">
    <div class="chart-card"><h3>Latence moyenne (secondes)</h3><p class="chart-sub">avg_latency_s · plus bas = meilleur</p><div id="legLatency" class="legend"></div><div style="position:relative;height:260px"><canvas id="chartLatency"></canvas></div></div>
  </div>
  <div class="section-title">Vue scatter — Accuracy vs Latence</div>
  <div class="chart-full">
    <h3>Accuracy vs Latence</h3>
    <p class="chart-sub">X = latence (s) · Y = GT Accuracy (%) · taille bulle = fill rate · idéal = haut-gauche</p>
    <div id="legScatter" class="legend"></div>
    <div style="position:relative;height:320px"><canvas id="chartScatter"></canvas></div>
  </div>
  <div class="section-title">Données détaillées</div>
  <div class="table-wrap">
    <div class="table-header"><h3>Tous les résultats</h3><span id="rowCount" style="font-size:12px;font-family:var(--mono);color:var(--muted)"></span></div>
    <div style="overflow-x:auto">
      <table><thead><tr>
        <th>#</th>
        <th onclick="sortTable('model')">Modèle ↕</th>
        <th onclick="sortTable('parser')">Parseur ↕</th>
        <th onclick="sortTable('gt_accuracy_pct')">GT Accuracy ↕</th>
        <th onclick="sortTable('fill_rate_pct')">Fill Rate ↕</th>
        <th onclick="sortTable('avg_latency_s')">Latence (s) ↕</th>
      </tr></thead>
      <tbody id="tableBody"></tbody></table>
    </div>
  </div>
</div>

<footer class="footer">
  <button class="footer-toggle" id="footerToggle" onclick="toggleFooter()">
    <span>⚙ Spécifications techniques du benchmark</span>
    <span class="arrow">▾</span>
  </button>
  <div class="footer-body" id="footerBody"></div>
</footer>

<script>
const RAW_DATA = __DATA_JSON__;
const SPECS    = __SPECS_JSON__;
const COLORS = ['#4f9cf9','#7b6ef6','#2dd4bf','#fbbf24','#f87171','#4ade80','#fb923c','#e879f9'];
let parsers = [...new Set(RAW_DATA.map(r=>r.parser))];
let activeParsers = new Set(parsers);
let ocrFilter = 'all';
let tableSortCol = 'gt_accuracy_pct', tableSortDir = -1;
const charts = {};

function col(p,a=1){const h=COLORS[parsers.indexOf(p)%COLORS.length];if(a===1)return h;const r=parseInt(h.slice(1,3),16),g=parseInt(h.slice(3,5),16),b=parseInt(h.slice(5,7),16);return `rgba(${r},${g},${b},${a})`;}
function fmt(v){return v==null?'—':typeof v==='number'?v.toFixed(1):v;}
function filtered(){return RAW_DATA.filter(r=>activeParsers.has(r.parser)&&(ocrFilter==='all'||r.model===ocrFilter));}

function buildToggles(){
  document.getElementById('parserToggles').innerHTML=parsers.map((p,i)=>`
    <button class="parser-toggle active" data-i="${i}" onclick="toggleP('${p}',this,${i})"
      style="background:${COLORS[i]};border-color:${COLORS[i]};color:#0d0f14">${p}</button>`).join('');
}
function toggleP(p,btn,i){
  if(activeParsers.has(p)){if(activeParsers.size===1)return;activeParsers.delete(p);btn.classList.remove('active');btn.style.cssText=`background:transparent;color:var(--muted);border-color:var(--border2)`;}
  else{activeParsers.add(p);btn.classList.add('active');btn.style.cssText=`background:${COLORS[i]};color:#0d0f14;border-color:${COLORS[i]}`;}
  updateAll();
}
function buildOcrFilter(){
  const models=[...new Set(RAW_DATA.map(r=>r.model))].sort();
  const sel=document.getElementById('ocrFilter');
  sel.innerHTML='<option value="all">Tous</option>'+models.map(m=>`<option value="${m}">${m}</option>`).join('');
  sel.onchange=()=>{ocrFilter=sel.value;updateAll();};
}
function resetFilters(){
  activeParsers=new Set(parsers);ocrFilter='all';
  document.getElementById('ocrFilter').value='all';
  document.querySelectorAll('.parser-toggle').forEach((b,i)=>{b.classList.add('active');b.style.cssText=`background:${COLORS[i]};color:#0d0f14;border-color:${COLORS[i]}`;});
  updateAll();
}
document.getElementById('sortBy').onchange=function(){
  tableSortCol=this.value;
  tableSortDir=this.value==='avg_latency_s'?1:-1;
  buildTable(filtered());
};

function buildKPIs(data){
  if(!data.length)return;

  // KPI 1 — Meilleur GT Accuracy
  const bestAcc = data.reduce((a,b)=>(b.gt_accuracy_pct||0)>(a.gt_accuracy_pct||0)?b:a);

  // KPI 2 — Meilleur Fill Rate
  const bestFill = data.reduce((a,b)=>(b.fill_rate_pct||0)>(a.fill_rate_pct||0)?b:a);

  // KPI 3 — Modèle le plus rapide (latence min)
  const fastest = data.reduce((a,b)=>(b.avg_latency_s!=null&&(a.avg_latency_s==null||b.avg_latency_s<a.avg_latency_s))?b:a);

  // KPI 4 — Pire latence (à éviter)
  const slowest = data.reduce((a,b)=>(b.avg_latency_s!=null&&(a.avg_latency_s==null||b.avg_latency_s>a.avg_latency_s))?b:a);

  document.getElementById('kpiRow').innerHTML=`
    <div class="kpi-card c-green">
      <div class="label">🏆 Meilleur GT Accuracy</div>
      <div class="value">${fmt(bestAcc.gt_accuracy_pct)}%</div>
      <div class="sub">${bestAcc.model} · ${bestAcc.parser}</div>
    </div>
    <div class="kpi-card c-blue">
      <div class="label">📋 Meilleur Fill Rate</div>
      <div class="value">${fmt(bestFill.fill_rate_pct)}%</div>
      <div class="sub">${bestFill.model} · ${bestFill.parser}</div>
    </div>
    <div class="kpi-card c-teal">
      <div class="label">⚡ Plus rapide</div>
      <div class="value">${fmt(fastest.avg_latency_s)}s</div>
      <div class="sub">${fastest.model} · ${fastest.parser}</div>
    </div>
    <div class="kpi-card c-red">
      <div class="label">🐢 Pire latence</div>
      <div class="value">${fmt(slowest.avg_latency_s)}s</div>
      <div class="sub">${slowest.model} · ${slowest.parser}</div>
    </div>`;
}

function legend(id,pArr){document.getElementById(id).innerHTML=pArr.map(p=>`<span class="legend-item"><span class="legend-dot" style="background:${col(p)}"></span>${p}</span>`).join('');}

const tick={color:'#6b7080',font:{size:11,family:"'Space Mono',monospace"}};
const grid={color:'rgba(255,255,255,0.05)'};
const baseOpts={responsive:true,maintainAspectRatio:false,plugins:{legend:{display:false},tooltip:{bodyFont:{family:"'Space Mono',monospace",size:12}}},scales:{x:{ticks:{...tick,maxRotation:35},grid},y:{ticks:tick,grid}}};

function buildCharts(data){
  const activePArr=[...activeParsers].filter(p=>data.some(r=>r.parser===p));
  const models=[...new Set(data.map(r=>r.model))].sort();
  function ds(metric){return activePArr.map(p=>({label:p,data:models.map(m=>{const r=data.find(d=>d.model===m&&d.parser===p);return r?(r[metric]??null):null;}),backgroundColor:col(p,.75),borderColor:col(p),borderWidth:1.5,borderRadius:3}));}
  ['chartAccuracy','chartFill','chartLatency','chartScatter'].forEach(id=>{if(charts[id]){charts[id].destroy();delete charts[id];}});
  charts.chartAccuracy=new Chart(document.getElementById('chartAccuracy'),{type:'bar',data:{labels:models,datasets:ds('gt_accuracy_pct')},options:{...baseOpts,scales:{...baseOpts.scales,y:{...baseOpts.scales.y,max:100}}}});
  legend('legAccuracy',activePArr);
  charts.chartFill=new Chart(document.getElementById('chartFill'),{type:'bar',data:{labels:models,datasets:ds('fill_rate_pct')},options:{...baseOpts,scales:{...baseOpts.scales,y:{...baseOpts.scales.y,max:100}}}});
  legend('legFill',activePArr);
  charts.chartLatency=new Chart(document.getElementById('chartLatency'),{type:'bar',data:{labels:models,datasets:ds('avg_latency_s')},options:baseOpts});
  legend('legLatency',activePArr);
  charts.chartScatter=new Chart(document.getElementById('chartScatter'),{type:'bubble',
    data:{datasets:activePArr.map(p=>({label:p,data:data.filter(r=>r.parser===p).map(r=>({x:r.avg_latency_s,y:r.gt_accuracy_pct,r:Math.max(5,(r.fill_rate_pct/100)*18),model:r.model,fill:r.fill_rate_pct})),backgroundColor:col(p,.5),borderColor:col(p),borderWidth:1.5}))},
    options:{responsive:true,maintainAspectRatio:false,plugins:{legend:{display:false},tooltip:{callbacks:{label:c=>`${c.raw.model} (${c.dataset.label}) acc:${c.raw.y}% lat:${c.raw.x}s fill:${c.raw.fill}%`},bodyFont:{family:"'Space Mono',monospace",size:11}}},
    scales:{x:{title:{display:true,text:'Latence (s)',color:'#6b7080'},ticks:tick,grid},y:{title:{display:true,text:'GT Accuracy (%)',color:'#6b7080'},ticks:tick,grid,min:0,max:105}}}});
  legend('legScatter',activePArr);
}

function sortTable(c){if(tableSortCol===c)tableSortDir*=-1;else{tableSortCol=c;tableSortDir=-1;}buildTable(filtered());}
function buildTable(data){
  const sorted=[...data].sort((a,b)=>{const av=a[tableSortCol]??'',bv=b[tableSortCol]??'';return(typeof av==='number'?(av-bv):String(av).localeCompare(String(bv)))*tableSortDir;});
  document.getElementById('rowCount').textContent=`${sorted.length} lignes`;
  const maxV=k=>Math.max(...data.map(r=>r[k]||0));
  const [mA,mF,mL]=['gt_accuracy_pct','fill_rate_pct','avg_latency_s'].map(maxV);
  function bar(v,max,c){const p=max?(v/max*100):0;return `<div class="bar-cell"><span class="td-val">${fmt(v)}</span><div class="bar-track"><div class="bar-fill" style="width:${p}%;background:${c}"></div></div></div>`;}
  document.getElementById('tableBody').innerHTML=sorted.map((r,i)=>{
    const pi=parsers.indexOf(r.parser),c=COLORS[pi%COLORS.length],rc=i<3?`rank-${i+1}`:'rank-other';
    return `<tr><td><span class="rank-badge ${rc}">${i+1}</span></td><td class="td-model">${r.model}</td><td><span class="td-parser" style="background:${c}22;color:${c};border:1px solid ${c}44">${r.parser}</span></td><td>${bar(r.gt_accuracy_pct,mA,'#4ade80')}</td><td>${bar(r.fill_rate_pct,mF,'#4f9cf9')}</td><td>${bar(r.avg_latency_s,mL,'#fbbf24')}</td></tr>`;
  }).join('');
}
function updateAll(){const d=filtered();buildKPIs(d);buildCharts(d);buildTable(d);}

function toggleFooter(){
  const btn=document.getElementById('footerToggle');
  const body=document.getElementById('footerBody');
  btn.classList.toggle('open');
  body.classList.toggle('open');
}

function buildFooter(){
  const s = SPECS;
  if(!s || !Object.keys(s).length) return;
  function row(k,v,cls=''){return `<div class="spec-row"><span class="spec-key">${k}</span><span class="spec-val ${cls}">${v||'—'}</span></div>`;}
  const envFields=[['date',s.date,'accent'],['os',s.os,''],['python',s.python_version,'teal'],['cpu',s.cpu,''],['gpu',s.gpu,'warning'],['ram',s.ram,''],['pipeline',s.pipeline_version,'accent']].filter(([,v])=>v);
  const evalFields=[['dataset',s.dataset,''],['split',s.dataset_split,''],['métriques',s.eval_metric,'teal'],['runs',s.n_runs,'']].filter(([,v])=>v);
  const pkgs=s.packages?Object.entries(s.packages):[];
  const ocrModels=s.ocr_models?Object.entries(s.ocr_models):[];
  const llmParsers=s.llm_parsers?Object.entries(s.llm_parsers):[];
  let html='<div class="specs-grid">';
  if(envFields.length)html+=`<div class="spec-card"><h4>🖥 Environnement</h4>${envFields.map(([k,v,c])=>row(k,v,c)).join('')}</div>`;
  if(evalFields.length)html+=`<div class="spec-card"><h4>📐 Dataset &amp; Évaluation</h4>${evalFields.map(([k,v,c])=>row(k,v,c)).join('')}</div>`;
  if(pkgs.length)html+=`<div class="spec-card"><h4>📦 Packages Python</h4><div class="pkg-grid">${pkgs.map(([n,v])=>`<div class="pkg-row"><span class="pkg-name">${n}</span><span class="pkg-ver">${v}</span></div>`).join('')}</div></div>`;
  if(ocrModels.length)html+=`<div class="spec-card"><h4>🔍 Modèles OCR testés</h4>${ocrModels.map(([k,v])=>`<div class="model-row"><span class="model-key">${k}</span><span class="model-desc">${v}</span></div>`).join('')}</div>`;
  if(llmParsers.length)html+=`<div class="spec-card"><h4>🤖 Parseurs LLM testés</h4>${llmParsers.map(([k,v])=>`<div class="model-row"><span class="model-key">${k}</span><span class="model-desc">${v}</span></div>`).join('')}</div>`;
  html+='</div>';
  if(s.notes)html+=`<div class="notes-box">📝 &nbsp;${s.notes}</div>`;
  document.getElementById('footerBody').innerHTML=html;
}

buildToggles();
buildOcrFilter();
document.getElementById('headerSub').textContent=`// ${parsers.length} parseur(s) · ${RAW_DATA.length} entrées`;
document.getElementById('headerBadge').textContent=`${parsers.length} parseurs chargés`;
updateAll();
buildFooter();
</script>
</body></html>
"""

generate_dashboard(df, OUTPUT_HTML, BENCHMARK_SPECS)

## 🚀 Ouvrir le dashboard

In [ ]:
from IPython.display import IFrame, display, HTML

# Affichage inline dans le notebook
display(HTML(f'<a href="{OUTPUT_HTML}" target="_blank" style="font-size:15px;font-weight:bold">🌐 Ouvrir le dashboard dans un nouvel onglet</a>'))

# Aperçu inline (peut être limité selon votre env Jupyter)
IFrame(src=OUTPUT_HTML, width='100%', height=900)